# Масштабирование и преобразование признаков в кибербезопасности

**Рабочая тетрадь студента**  
Практическое занятие — 80 минут. Данные синтетические и предназначены только для обучения.

## Правила работы

1. Преобразователи обучаются только на `train`, к `test` применяется только `transform`.
2. `dst_port` является числовым кодом категории и автоматически не масштабируется.
3. Линейное масштабирование не меняет форму распределения.
4. После каждого преобразования оцениваются технические статистики и смысл для SOC.

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import (StandardScaler, RobustScaler, MinMaxScaler, MaxAbsScaler, Normalizer, PowerTransformer)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

DATA_PATH = 'soc_feature_scaling_lab.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['event_time'])

train = df[df['split'].eq('train')].copy()
test = df[df['split'].eq('test')].copy()
print('Размер:', df.shape, '| train:', len(train), '| test:', len(test))
df.head()

## Задание 1. Диагностика масштаба и распределений (3 балла)

Для числовых признаков рассчитайте минимум, максимум, медиану, стандартное отклонение, IQR, асимметрию и эксцесс. Постройте гистограммы `response_time_ms`, `duration_sec`, `bytes_sent`, `payload_entropy`. Определите признаки с естественными границами, тяжёлым хвостом, выбросами и отрицательными значениями. Объясните, почему `dst_port` нельзя трактовать как обычный непрерывный признак.

In [ ]:
numeric = ['duration_sec','bytes_sent','n_packets','response_time_ms','payload_entropy','failed_logins','auth_balance']
# таблица min/max/median/std/IQR/skew/kurtosis
# гистограммы и интерпретация

## Задание 2. Сравнение StandardScaler, RobustScaler и MinMaxScaler (5 баллов)

Обучите все преобразователи только на `train`. Для `bytes_sent` сравните StandardScaler и RobustScaler: медиану, IQR, 95-й и 99-й процентили, максимум и положение обычных наблюдений относительно атак. Для `payload_entropy` примените MinMaxScaler. Покажите, могут ли тестовые значения выйти за [0, 1]. Сформулируйте правило выбора метода при выбросах и фиксированных физических границах.

In [ ]:
# fit только на train[['bytes_sent']], transform для train и test
# StandardScaler и RobustScaler: сравните статистики и классы attack_type

In [ ]:
# MinMaxScaler для payload_entropy
# Проверьте min/max test и значения вне [0, 1]

## Задание 3. Разреженные данные и нормализация строк (3 балла)

Для трёх TF-IDF-признаков вычислите долю нулей до и после MaxAbsScaler и подтвердите сохранение разреженности. Затем примените L2 Normalizer к строкам и проверьте нормы ненулевых строк. Объясните, когда сравнение направления профиля полезно для UEBA и почему L2-нормализация опасна для DDoS-детектора.

In [ ]:
sparse_cols = ['tfidf_error','tfidf_timeout','tfidf_admin']
# MaxAbsScaler, доли нулей, L2 Normalizer, нормы строк

## Задание 4. Нелинейные преобразования (5 баллов)

Сравните для `bytes_sent` исходный признак, `log1p`, квадратный корень и Box-Cox по асимметрии и эксцессу. Для `auth_balance`, содержащего отрицательные значения, примените Yeo-Johnson. Визуализируйте распределения. Обоснуйте, почему логарифмирование дополняется масштабированием и почему Box-Cox нельзя применять к нулям или отрицательным значениям.

In [ ]:
# log1p, sqrt и PowerTransformer(method='box-cox', standardize=False) для bytes_sent
# PowerTransformer(method='yeo-johnson', standardize=False) для auth_balance
# Сравните skew/kurtosis и гистограммы

## Задание 5. Безопасный конвейер и качество регрессии (4 балла)

Сравните Ridge-регрессию для прогноза `severity_score`: базовый вариант с исходными признаками и StandardScaler; вариант с `log1p(bytes_sent)`, `1/(duration_sec+0.55)`, `log1p(n_packets)`, `log1p(failed_logins)` и RobustScaler. Во втором варианте масштабируйте целевой признак через TransformedTargetRegressor, чтобы прогноз автоматически возвращался в исходную шкалу. Рассчитайте MAE, RMSE и R², выберите вариант и оформите записку SOC.

In [ ]:
features = ['bytes_sent','duration_sec','n_packets','response_time_ms','payload_entropy','failed_logins','auth_balance']
X_train, X_test = train[features], test[features]
y_train, y_test = train['severity_score'], test['severity_score']
# baseline Pipeline(StandardScaler, Ridge)
# функция feature_engineering и модель RobustScaler + Ridge

In [ ]:
# MAE, RMSE, R2 для двух вариантов
# проверка диапазона прогнозов в исходной шкале
# записка SOC из 6–8 предложений

## Самопроверка

Ноутбук должен выполняться сверху вниз после перезапуска ядра; каждый числовой результат сопровождается интерпретацией для SOC.